# 📰 Fake News Detection — NLP & Machine Learning

## 1. Introduction & Problem Statement

The spread of misinformation ("fake news") through online platforms is a
serious societal problem. **The goal of this project is to build a machine
learning model that automatically classifies a news article as either FAKE
(`0`) or REAL (`1`)** based purely on its text.

We use the **WELFake** dataset (columns: `title`, `text`, `label`) and the
following pipeline:

1. Text preprocessing & normalization (NLTK)
2. TF-IDF feature extraction (unigrams + bigrams)
3. Training & comparing two classifiers — Logistic Regression and
   Multinomial Naive Bayes
4. Evaluation (accuracy, precision, recall, F1, confusion matrix)
5. Feature-importance / explainability analysis

This notebook documents the full exploratory and modeling workflow.


In [ ]:
# Make the project's src/ package importable from the notebooks/ folder.
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_colwidth", 120)
print("Imports ready.")

## 2. Data Loading & Exploration

We load the raw dataset and inspect its shape, class distribution, and a few
sample rows. Update `DATA_PATH` if your CSV lives elsewhere.


In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "WELFake_Dataset.csv")

df = pd.read_csv(DATA_PATH)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]  # drop stray index column
print("Shape:", df.shape)
df.head()

In [ ]:
# Basic info and missing values
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# Class distribution (0 = fake, 1 = real)
counts = df["label"].value_counts().sort_index()
print(counts)
print("\nProportions:")
print((counts / counts.sum()).round(3))

## 3. Exploratory Data Analysis (EDA)

### 3.1 Class distribution


In [ ]:
ax = sns.countplot(x="label", data=df, palette=["#dc2626", "#16a34a"])
ax.set_title("Class Distribution (0 = Fake, 1 = Real)")
ax.set_xlabel("Label")
ax.set_ylabel("Number of articles")
ax.set_xticklabels(["Fake (0)", "Real (1)"])
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout(); plt.show()

### 3.2 Article length distribution

In [ ]:
# Combine title + text, then measure word counts.
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["content"] = (df["title"] + " " + df["text"]).str.strip()
df["word_count"] = df["content"].str.split().apply(len)

plt.figure(figsize=(11, 5))
for label, color, name in [(0, "#dc2626", "Fake"), (1, "#16a34a", "Real")]:
    subset = df[df["label"] == label]["word_count"]
    # Clip extreme outliers for readability.
    sns.histplot(subset.clip(upper=2000), bins=60, color=color, label=name,
                 alpha=0.55, stat="density")
plt.title("Article Length Distribution (words, clipped at 2000)")
plt.xlabel("Word count"); plt.ylabel("Density"); plt.legend()
plt.tight_layout(); plt.show()

print(df.groupby("label")["word_count"].describe())

### 3.3 Word Clouds — Fake vs Real

We clean the text first using the project's shared preprocessing function so
the word clouds reflect exactly what the model sees.


In [ ]:
from src.preprocess import clean_text

# Clean a sample for speed (use the full set if you have time/RAM).
SAMPLE_N = 4000
sample_df = df.sample(min(SAMPLE_N, len(df)), random_state=42).copy()
sample_df["clean"] = sample_df["content"].apply(clean_text)

fake_text = " ".join(sample_df[sample_df["label"] == 0]["clean"])
real_text = " ".join(sample_df[sample_df["label"] == 1]["clean"])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, text, title, cmap in [
    (axes[0], fake_text, "Fake News", "Reds"),
    (axes[1], real_text, "Real News", "Greens"),
]:
    wc = WordCloud(width=800, height=500, background_color="white",
                   colormap=cmap, max_words=150).generate(text or "news")
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
    ax.set_title(f"{title} — Word Cloud", fontsize=15)
plt.tight_layout(); plt.show()

### 3.4 Top 20 most frequent words per class

In [ ]:
from collections import Counter

def top_words(series, n=20):
    words = " ".join(series).split()
    return Counter(words).most_common(n)

fake_top = top_words(sample_df[sample_df["label"] == 0]["clean"])
real_top = top_words(sample_df[sample_df["label"] == 1]["clean"])

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, data, title, color in [
    (axes[0], fake_top, "Top words — Fake", "#dc2626"),
    (axes[1], real_top, "Top words — Real", "#16a34a"),
]:
    words, freqs = zip(*data)
    sns.barplot(x=list(freqs), y=list(words), ax=ax, color=color)
    ax.set_title(title); ax.set_xlabel("Frequency")
plt.tight_layout(); plt.show()

## 4. Text Preprocessing Walkthrough

The `clean_text` function applies: lowercasing → URL removal → HTML stripping →
punctuation removal → number removal → tokenization → stopword removal →
lemmatization. Here is a before/after example.


In [ ]:
example = df["content"].iloc[0]
print("BEFORE:\n", example[:500], "\n")
print("AFTER:\n", clean_text(example)[:500])

## 5. TF-IDF Vectorization

**TF-IDF** (Term Frequency–Inverse Document Frequency) converts text into
numerical vectors. Words that are frequent in a document but rare across the
corpus get higher weights — capturing what makes a document distinctive.

Configuration used in this project:
- `max_features=50000` — cap the vocabulary to the 50k most informative terms
- `ngram_range=(1, 2)` — include single words **and** two-word phrases
- `sublinear_tf=True` — apply `1 + log(tf)` scaling to dampen very frequent terms


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Clean the full dataset (this is the expensive step).
df["clean_content"] = df["content"].apply(clean_text)
df = df[df["clean_content"].str.len() > 0].reset_index(drop=True)

X_raw = df["clean_content"]
y = df["label"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)
print("TF-IDF train matrix:", X_train.shape)

## 6. Model Training — Logistic Regression & Naive Bayes

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

logreg = LogisticRegression(C=1.0, max_iter=1000)
logreg.fit(X_train, y_train)

nb = MultinomialNB()
nb.fit(X_train, y_train)

print("Both models trained.")

## 7. Evaluation — Classification Report & Confusion Matrix

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

def evaluate(name, model):
    preds = model.predict(X_test)
    print(f"===== {name} =====")
    print(classification_report(y_test, preds, target_names=["Fake", "Real"]))
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "preds": preds,
    }

res_lr = evaluate("Logistic Regression", logreg)
res_nb = evaluate("Multinomial Naive Bayes", nb)

comparison = pd.DataFrame([res_lr, res_nb]).drop(columns="preds").set_index("Model")
comparison.round(4)

In [ ]:
# Confusion-matrix heatmaps
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, res in zip(axes, [res_lr, res_nb]):
    cm = confusion_matrix(y_test, res["preds"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
    ax.set_title(res["Model"]); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

## 8. Feature Importance — Words Most Associated with Fake vs Real

For Logistic Regression, the sign and magnitude of each feature's coefficient
tells us how strongly a word pushes a prediction toward **REAL** (positive) or
**FAKE** (negative).


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = logreg.coef_[0]

top_real_idx = np.argsort(coefs)[-20:][::-1]
top_fake_idx = np.argsort(coefs)[:20]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
sns.barplot(x=coefs[top_real_idx], y=feature_names[top_real_idx],
            ax=axes[0], color="#16a34a")
axes[0].set_title("Top 20 words → REAL"); axes[0].set_xlabel("Coefficient")

sns.barplot(x=coefs[top_fake_idx], y=feature_names[top_fake_idx],
            ax=axes[1], color="#dc2626")
axes[1].set_title("Top 20 words → FAKE"); axes[1].set_xlabel("Coefficient")
plt.tight_layout(); plt.show()

## 9. Conclusion

- We built a complete fake-news classification pipeline: **NLTK preprocessing →
  TF-IDF features → linear/Bayesian classifiers**.
- **Logistic Regression** generally outperforms Multinomial Naive Bayes on this
  dataset, typically reaching ~0.95–0.96 F1.
- Feature-importance analysis shows the model keys on intuitive signals
  (sensational/clickbait vocabulary vs. measured, source-attributed language).

**Next steps / improvements:**
- Try linear SVM or gradient-boosted trees, and calibrated probabilities.
- Add transformer embeddings (e.g. DistilBERT) for richer semantics.
- Evaluate on out-of-distribution / more recent news to test generalization.
- Monitor for dataset bias (topic/temporal leakage) before any real deployment.

The trained model and vectorizer are saved by `src/train.py` and served through
the Streamlit app in `app/streamlit_app.py`.
